In [6]:
!nvidia-smi

Sun Apr 19 14:27:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Import required libraries

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim  # contains optimizers
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy
import random
import matplotlib.pyplot as plt

In [5]:
torch.__version__

'2.10.0+cu128'

## 2. Setup Device-Agnostic Code

In [10]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [11]:
print(f"Using Device: {device}")

Using Device: cuda


##  3. Set the seed

In [12]:
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
random.seed(42)

## 4. Setting the Hyperparameters

In [23]:
BATCH_SIZE = 128
EPOCHS = 10
LEARNING_RATE = 3e-4
PATCH_SIZE = 4
NUM_CLASSES = 10
IMAGE_SIZE = 32
CHANNELS = 3
EMBED_DIM = 256  # ?
NUM_HEADS = 8  # ?
DEPTH = 6  # ?
MLP_DIM = 512  # ?
DROP_RATE = 0.1  # ?

## 5. Define Image Transformations

In [15]:
transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize((0.5), (0.5)),
        # 1. Helps the model to converge faster
        # 2. Helps to make the numerical computations stable
    ]
)

## 6. Getting Dataset

In [16]:
train_dataset = datasets.CIFAR10(
    root="data", train=True, download=True, transform=transform
)

100%|██████████| 170M/170M [00:16<00:00, 10.4MB/s] 


In [17]:
test_dataset = datasets.CIFAR10(
    root="data", train=False, download=True, transform=transform
)

In [19]:
train_dataset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=0.5, std=0.5)
           )

In [20]:
test_dataset

Dataset CIFAR10
    Number of datapoints: 10000
    Root location: data
    Split: Test
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=0.5, std=0.5)
           )

## 7. Datasets to Dataloaders

rn, our data is in the form of PyTorch datasets.

DataLoader turns our data into batches or (mini-batches)

Why would we do this?

1. It is more computationally efficient, as in, your computing hardware may not be able to look (store in memroy) at 50000 images in one
hit. So we break it into 128 images at a time. (batch size of 128).
2. It gives our neural network more chances to update its gradients per epoch

In [21]:
train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(
    dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False
)  # Don't shuffle dataset for best practices

In [22]:
print(f"Dataloaders: {train_loader, test_loader}")
print(f"Length Train Loaders: {len(train_loader)} batches of {BATCH_SIZE}")
print(f"Length Test Loaders: {len(test_loader)} batches of {BATCH_SIZE}")

Dataloaders: (<torch.utils.data.dataloader.DataLoader object at 0x7d315f87eba0>, <torch.utils.data.dataloader.DataLoader object at 0x7d315f791670>)
Length Train Loaders: 391 batches of 128
Length Test Loaders: 79 batches of 128


## 8. Building Vision Transformer Model from Scratch

In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, img_size, patch_size, channels, embed_dim):
        super().__init__()  # It is necessary so PyTorch can track all the parameters and submodules ...
        self.patch_size = patch_size
        self.proj = nn.Conv2d(
            in_channels=channels,
            out_channels=embed_dim,
            kernel_size=patch_size,
            stride=patch_size,
        )
        num_patches = (img_size // patch_size) ** 2
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.randn(1, 1 + num_patches, embed_dim))

    def forward(self, x: torch.Tensor):
        # Input x: [B, C, H, W] = [32, 3, 32, 32] (batch of CIFAR images)
        B = x.size(0)

        x = self.proj(x)  # (B, E, H/P_size, W/P_size)
        # self.proj = Conv2d(channels=3, embed_dim=256, kernel=4, stride=4)
        # Input:  [32, 3,  32, 32]  ← 32x32 RGB images
        # Output: [32, 256, 8,  8]   ← 8x8 grid of 256-dim patch embeddings

        x = x.flatten(2).transpose(
            1, 2
        )  # (B, N, E) where N=num_patches and E=embed_dim
        cls_token = self.cls_token.expand(B, -1, -1)  # Expand to match batch size
        x = torc.cat((cls_token, x), dim=1)
        x = x + self.pos_embed
        return x